In [23]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import numpy as np
from xgboost.callback import EarlyStopping

In [24]:
def featurize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Skip invalid SMILES

    return {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumRings': Descriptors.RingCount(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumAtoms': mol.GetNumAtoms(),
        'NumBonds': mol.GetNumBonds(),
    }



In [31]:
df = pd.read_csv("data/train.csv")

In [32]:
train

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,NaN,0.378860,NaN,NaN,NaN
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,NaN,0.387324,NaN,NaN,NaN
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,NaN,0.355470,NaN,NaN,NaN
...,...,...,...,...,...,...,...
7968,2146592435,*Oc1cc(CCCCCCCC)cc(OC(=O)c2cccc(C(*)=O)c2)c1,NaN,0.367498,NaN,NaN,NaN
7969,2146810552,*C(=O)OCCN(CCOC(=O)c1ccc2c(c1)C(=O)N(c1cccc(N3...,NaN,0.353280,NaN,NaN,NaN
7970,2147191531,*c1cc(C(=O)NCCCCCCCC)cc(N2C(=O)c3ccc(-c4ccc5c(...,NaN,0.369411,NaN,NaN,NaN
7971,2147435020,*C=C(*)c1ccccc1C,261.662355,NaN,NaN,NaN,NaN


In [33]:
feature_list = []
valid_indices = []
for idx, smi in enumerate(df['SMILES']):
    features = featurize_smiles(smi)
    if features:
        feature_list.append(features)
        valid_indices.append(idx)

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

# === Load your dataset ===
# Replace with your actual CSV file

# === Feature Engineering from SMILES ===
def featurize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Skip invalid SMILES

    return {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumRings': Descriptors.RingCount(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumAtoms': mol.GetNumAtoms(),
        'NumBonds': mol.GetNumBonds(),
    }

# === Apply featurization ===
feature_list = []
valid_indices = []

for idx, smi in enumerate(df['SMILES']):
    features = featurize_smiles(smi)
    if features:
        feature_list.append(features)
        valid_indices.append(idx)

features_df = pd.DataFrame(feature_list)
df = df.loc[valid_indices].reset_index(drop=True)
df = pd.concat([df, features_df], axis=1)

# === Define target columns ===
target_cols = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
X_all = df[features_df.columns]

# === Model storage ===
models = {}
results = {}

# === Train one model per target ===
for col in target_cols:
    df_sub = df[df[col].notna()]  # Only keep rows with valid target values
    if df_sub.shape[0] < 20:
        print(f"Skipping {col} due to too few samples.")
        continue

    X = df_sub[features_df.columns]
    y = df_sub[col]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Convert to DMatrix for native API
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    # Define XGBoost parameters
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "seed": 42
    }

    evallist = [(dtrain, "train"), (dval, "eval")]

    # Train with early stopping
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=300,
        evals=evallist,
        early_stopping_rounds=20,
        verbose_eval=False
    )

    # Predict and evaluate
    y_pred = model.predict(dval)
    mae = mean_absolute_error(y_val, y_pred)

    print(f"{col} MAE: {mae:.4f}")

    # Store results
    models[col] = model
    results[col] = {
        "mae": mae,
        "y_val": y_val,
        "y_pred": y_pred
    }


Tg MAE: 56.9990
FFV MAE: 0.0115
Tc MAE: 0.0331
Density MAE: 0.0605
Rg MAE: 2.8749


In [35]:
def compute_wmae(results_dict, full_df, target_cols):
    """
    Compute weighted MAE based on your validation predictions.
    Args:
        results_dict: output from model loop (with y_val and y_pred per property)
        full_df: original full DataFrame with targets
        target_cols: list of 5 property names
    Returns:
        wMAE score (float)
    """
    K = len(target_cols)
    n_i = {}
    r_i = {}
    raw_mae = {}
    y_true_all = {}
    y_pred_all = {}

    for col in target_cols:
        if col not in results_dict:
            continue
        y_true = results_dict[col]['y_val']
        y_pred = results_dict[col]['y_pred']
        y_true_all[col] = y_true
        y_pred_all[col] = y_pred
        n_i[col] = len(y_true)
        r_i[col] = full_df[col].max() - full_df[col].min()
        raw_mae[col] = mean_absolute_error(y_true, y_pred)

    # Inverse sqrt scaling
    sqrt_inv_ni = {col: np.sqrt(1 / n_i[col]) for col in n_i}
    denom = sum(sqrt_inv_ni.values())
    scaling_factors = {col: (K * sqrt_inv_ni[col] / denom) for col in n_i}

    # Final weights
    weights = {col: (1 / r_i[col]) * scaling_factors[col] for col in n_i}

    # Weighted MAE
    wmae = sum(weights[col] * raw_mae[col] for col in n_i)

    print("\n--- Weighted MAE (wMAE) Breakdown ---")
    for col in n_i:
        print(f"{col}: MAE={raw_mae[col]:.4f}, weight={weights[col]:.4f}")
    print(f"\nFinal wMAE: {wmae:.5f}")

    return wmae


In [36]:
# Final wMAE result
wmae_score = compute_wmae(results, df, target_cols)



--- Weighted MAE (wMAE) Breakdown ---
Tg: MAE=56.9990, weight=0.0020
FFV: MAE=0.0115, weight=0.6252
Tc: MAE=0.0331, weight=2.2199
Density: MAE=0.0605, weight=1.0645
Rg: MAE=2.8749, weight=0.0466

Final wMAE: 0.39594


In [ ]:
asd